### Generate max_atoms config from a target SMILES

In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
from rdkit import Chem


def generateMaxAtomsConfig(targetSmiles: str, increasePercent: float = 0.5) -> dict:
    """
    Generate max_atoms config from a target SMILES.
    Increases each atom count by the specified percentage (default 50%).
    Returns a dictionary with atom limits.
    """
    mol = Chem.MolFromSmiles(targetSmiles)
    if mol is None:
        raise ValueError(f"Could not parse SMILES: {targetSmiles}")

    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())

    atomsOfInterest = ['C', 'N', 'O', 'S']

    maxAtoms = {}
    print(f"\nTarget molecule SMILES: {targetSmiles}")
    print(f"\nAtom counts in target molecule:")
    for atom in atomsOfInterest:
        count = atomCounter.get(atom, 0)
        suggested = int(np.ceil(count * (1 + increasePercent)))
        suggested = max(suggested, 1)
        maxAtoms[atom] = suggested
        print(f"  {atom}: {count}")

    print(f"\nGenerated max_atoms config ({int(increasePercent * 100)}% increase):")
    print(f"max_atoms:")
    for atom in atomsOfInterest:
        names = {'C': 'Carbon', 'N': 'Nitrogen', 'O': 'Oxygen', 'S': 'Sulfur'}
        print(f"  {atom}: {maxAtoms[atom]}   # {names[atom]}")

    return maxAtoms


if __name__ == "__main__":
    # ---- User supplies only this ----
    targetSmiles = "C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H](O3)CO)O)O"

    maxAtomsConfig = generateMaxAtomsConfig(targetSmiles)


Target molecule SMILES: C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H](O3)CO)O)O

Atom counts in target molecule:
  C: 10
  N: 4
  O: 5
  S: 0

Generated max_atoms config (50% increase):
max_atoms:
  C: 15   # Carbon
  N: 6   # Nitrogen
  O: 8   # Oxygen
  S: 1   # Sulfur


### Prepare config file

`Inosice` similar substructures has been downloaded from Pubchem (https://pubchem.ncbi.nlm.nih.gov/#query=CID135398641+structure&tab=substructure&cid=135398641). Now we are going to convert it to csv file to run `DORAnet` for all the possible substructures. 

In [2]:
from pathlib import Path

import pandas as pd
from rdkit import Chem


def sdfToDataFrame(sdfPath: str) -> pd.DataFrame:
    """
    Load an SDF file into a pandas DataFrame.
    - One row per molecule
    - Columns include: SDF properties + SMILES + InChIKey
    """
    supplier = Chem.SDMolSupplier(sdfPath, removeHs=False)
    rows = []

    for idx, mol in enumerate(supplier):
        if mol is None:
            continue

        rowDict = {"recordIndex": idx}

        # Add all SDF properties as columns
        for propName in mol.GetPropNames():
            rowDict[propName] = mol.GetProp(propName)

        # Add a few useful computed identifiers
        rowDict["smiles"] = Chem.MolToSmiles(mol, isomericSmiles=True)
        rowDict["inchiKey"] = Chem.inchi.MolToInchiKey(mol)

        rows.append(rowDict)

    DF = pd.DataFrame(rows)
    return DF


def canonicalizeSmiles(smilesStr: str) -> str | None:
    """
    Canonicalize a SMILES string with RDKit.
    Returns None if parsing fails or input is missing.
    """
    if smilesStr is None:
        return None

    smilesStr = str(smilesStr).strip()
    if smilesStr == "" or smilesStr.lower() == "nan":
        return None

    mol = Chem.MolFromSmiles(smilesStr)
    if mol is None:
        return None

    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)


def ensureSmilesFirst(DF: pd.DataFrame, targetSmiles: str) -> pd.DataFrame:
    """
    Ensure `targetSmiles` appears as the first row in DF (based on exact SMILES string match),
    inserting it if it is not already present.
    """
    if "smiles" not in DF.columns:
        raise KeyError("Expected a 'smiles' column, but it was not found in the DataFrame.")

    targetSmiles = targetSmiles.strip()

    # Exact-match check (after stripping whitespace)
    existingSmilesSet = set(DF["smiles"].astype(str).str.strip().tolist())
    if targetSmiles in existingSmilesSet:
        # If present but not first, move it to top
        mask = DF["smiles"].astype(str).str.strip().eq(targetSmiles)
        DFTop = DF.loc[mask]
        DFRest = DF.loc[~mask]
        return pd.concat([DFTop, DFRest], ignore_index=True)

    # If not present, create a new row and prepend it
    mol = Chem.MolFromSmiles(targetSmiles)
    if mol is None:
        raise ValueError("The target SMILES could not be parsed by RDKit.")

    newRow = {
        "recordIndex": -1,  # sentinel for inserted row
        "smiles": targetSmiles,
        "inchiKey": Chem.inchi.MolToInchiKey(mol),
    }

    # Ensure any additional columns exist in new row (fill with None)
    for colName in DF.columns:
        if colName not in newRow:
            newRow[colName] = None

    DFNew = pd.DataFrame([newRow], columns=DF.columns)
    return pd.concat([DFNew, DF], ignore_index=True)


def saveDfAsCsvSameName(sdfPath: str, DF: pd.DataFrame) -> str:
    sdfFilePath = Path(sdfPath)
    csvFilePath = sdfFilePath.with_suffix(".csv")
    DF.to_csv(csvFilePath, index=False, encoding="utf-8")
    return str(csvFilePath)


if __name__ == "__main__":
    sdfPath = "Inosine_PubChem_similarstructures.sdf"

    DF = sdfToDataFrame(sdfPath)

    # ---- Ensure this SMILES is first (insert if missing) ----
    targetSmiles = "C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H](O3)CO)O)O"  # Inosine SMILES
    DF = ensureSmilesFirst(DF, targetSmiles)

    # ---- Canonicalize SMILES into a new column ----
    DF["Canonical_SMILES"] = DF["smiles"].apply(canonicalizeSmiles)

    # ---- Save CSV ----
    csvPath = saveDfAsCsvSameName(sdfPath, DF)
    print(f"\nSaved CSV to: {csvPath}")


[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:06] WARNING: not removing hydrogen atom without neighbors
[12:31:07] WARNING: not removing hydrogen atom without neighbors
[12:31:07] WARNING: not removing hydrogen atom without neighbors
[12:31:07] WARNING: not r


Saved CSV to: Inosine_PubChem_similarstructures.csv


In [3]:
DF

,recordIndex,PUBCHEM_COMPOUND_CID,PUBCHEM_COMPOUND_CANONICALIZED,PUBCHEM_CACTVS_COMPLEXITY,PUBCHEM_CACTVS_HBOND_ACCEPTOR,PUBCHEM_CACTVS_HBOND_DONOR,PUBCHEM_CACTVS_ROTATABLE_BOND,PUBCHEM_CACTVS_SUBSKEYS,PUBCHEM_IUPAC_OPENEYE_NAME,PUBCHEM_IUPAC_CAS_NAME,...,PUBCHEM_ISOTOPIC_ATOM_COUNT,PUBCHEM_COMPONENT_COUNT,PUBCHEM_CACTVS_TAUTO_COUNT,PUBCHEM_COORDINATE_TYPE,PUBCHEM_BONDANNOTATIONS,smiles,inchiKey,PUBCHEM_XLOGP3_AA,PUBCHEM_NONSTANDARDBOND,Canonical_SMILES
0,-1,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,UGQMRVRMYYASKQ-KQYNXXCUSA-N,None,None,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...
1,0,135398640,1,555,10,5,4,AAADccBzvAIAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"[(2R,3S,4R,5R)-3,4-dihydroxy-5-(6-oxo-1H-purin...","[(2R,3S,4R,5R)-3,4-dihydroxy-5-(6-oxo-1H-purin...",...,0,1,-1,1\n5\n255,16 10 6\n10 19 8\n10 20 8\n11 20 8\n11...,[H]O[C@@]1([H])[C@@]([H])(O[H])[C@]([H])(n2c([...,GRSZFWQUAKGDAV-KQYNXXCUSA-N,NaN,NaN,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)O)[C...
2,1,135398641,1,405,7,4,2,AAADccBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxymethy...","9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxymethy...",...,0,1,-1,1\n5\n255,13 14 6\n15 17 8\n17 18 8\n10 2 5\n12 ...,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,UGQMRVRMYYASKQ-KQYNXXCUSA-N,NaN,NaN,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...
3,2,135398739,1,348,5,2,2,AAADccBzsAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"9-[(2R,5S)-5-(hydroxymethyl)tetrahydrofuran-2-...","9-[(2R,5S)-5-(hydroxymethyl)-2-oxolanyl]-1H-pu...",...,0,1,-1,1\n5\n255,11 12 6\n13 15 8\n15 16 8\n4 13 8\n4 ...,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,BXZVVICBKDXVGW-NKWVEPMBSA-N,NaN,NaN,O=c1[nH]cnc2c1ncn2[C@H]1CC[C@@H](CO)O1
4,3,135398635,1,446,7,5,2,AAADccBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"2-amino-9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydr...","2-amino-9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydr...",...,0,1,-1,1\n5\n255,14 15 5\n16 18 8\n18 19 8\n11 2 6\n13 ...,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,NYHBQMYGNKIUIF-UUOKFMHZSA-N,NaN,NaN,Nc1nc2c(ncn2[C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9112,9111,177857901,1,446,7,5,2,AAADccBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"2-amino-9-[(2S,3S,5S)-3,4-dihydroxy-5-(hydroxy...","2-amino-9-[(2S,3S,5S)-3,4-dihydroxy-5-(hydroxy...",...,0,1,-1,1\n5\n255,14 15 6\n16 18 8\n18 19 8\n11 2 5\n13 ...,[H]OC([H])([H])[C@]1([H])O[C@]([H])(n2c([H])nc...,NYHBQMYGNKIUIF-IASZTVIBSA-N,NaN,NaN,Nc1nc2c(ncn2[C@H]2O[C@@H](CO)C(O)[C@@H]2O)c(=O...
9113,9112,177857919,1,587,9,6,5,AAADceBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"2-[[9-[(3R,4S,5R)-3,4-dihydroxy-5-(hydroxymeth...","2-[[9-[(3R,4S,5R)-3,4-dihydroxy-5-(hydroxymeth...",...,0,1,-1,1\n5\n255,10 18 8\n10 22 8\n11 21 8\n11 22 8\n16...,[H]OC(=O)C([H])(N([H])c1nc2c(nc([H])n2C2([H])O...,JYXKXKDWVWQIOH-GZSQIZNNSA-N,NaN,NaN,CC(Nc1nc2c(ncn2C2O[C@H](CO)[C@@H](O)[C@H]2O)c(...
9114,9113,177857943,1,491,7,4,2,AAADccBzuQAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"2-amino-9-[(2R,3R,5R)-4-fluoro-3-hydroxy-5-(hy...","2-amino-9-[(2R,3R,5R)-4-fluoro-3-hydroxy-5-(hy...",...,0,1,-1,1\n5\n255,13 1 3\n14 16 6\n17 19 8\n19 20 8\n11 ...,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,FWLVLYHLIABSLF-RBXWHLPPSA-N,NaN,NaN,C[C@]1(O)C(F)[C@@H](CO)O[C@H]1n1cnc2c(=O)[nH]c...
9115,9114,177860646,1,899,16,4,7,AAADccBzvAMAAAAEAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"[[[(2R,3S,4R,5R)-5-(2-amino-6-oxo-1H-purin-9-y...","[[[(2R,3S,4R,5R)-5-(2-amino-6-oxo-1H-purin-9-y...",...,0,2,-1,1\n5\n255,26 19 5\n19 29 8\n19 30 8\n20 30 8\n20...,[Cu].[H]O[C@@]1([H])[C@@]([H])(O[H])[C@]([H])(...,JICNCLFSGJYKMS-GWTDSMLYSA-J,NaN,NaN,Nc1nc2c(ncn2[C@@H]2O[C@H](COP(=O)([O-])OP(=O)(...


### For a broader search space take `targetSmiles` from a data frame

In [4]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from collections import Counter
import pandas as pd
import numpy as np

# Read the CSV file
startersDF = DF
print(f"Loaded {len(startersDF)} molecules")

# Count atoms for each molecule
atomCounts = []

for smi in startersDF['Canonical_SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        atomCounts.append({'C': 0, 'N': 0, 'O': 0, 'S': 0})
        continue
    
    atomCounter = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    atomCounts.append({
        'C': atomCounter.get('C', 0),
        'N': atomCounter.get('N', 0),
        'O': atomCounter.get('O', 0),
        'S': atomCounter.get('S', 0)
    })

atomCountsDf = pd.DataFrame(atomCounts)
startersDF = pd.concat([startersDF, atomCountsDf], axis=1)

# Print atom count ranges
print(f"\nAtom count ranges across {len(startersDF)} molecules:")
print(f"  C (Carbon):   min = {startersDF['C'].min()}, max = {startersDF['C'].max()}")
print(f"  N (Nitrogen): min = {startersDF['N'].min()}, max = {startersDF['N'].max()}")
print(f"  O (Oxygen):   min = {startersDF['O'].min()}, max = {startersDF['O'].max()}")
print(f"  S (Sulfur):   min = {startersDF['S'].min()}, max = {startersDF['S'].max()}")

# Print suggested max_atoms config (max values + 50% increase)
maxC = startersDF['C'].max()
maxN = startersDF['N'].max()
maxO = startersDF['O'].max()
maxS = startersDF['S'].max()

print(f"\nSuggested max_atoms config (max values + 50% increase for expanded search space):")
print(f"  C: {int(np.ceil(maxC * 1.5))}")
print(f"  N: {int(np.ceil(maxN * 1.5))}")
print(f"  O: {int(np.ceil(maxO * 1.5))}")
print(f"  S: {int(np.ceil(maxS * 1.5))}")

startersDF

Loaded 9117 molecules


[12:32:06] WARNING: not removing hydrogen atom without neighbors
[12:32:06] WARNING: not removing hydrogen atom without neighbors
[12:32:06] WARNING: not removing hydrogen atom without neighbors
[12:32:06] WARNING: not removing hydrogen atom without neighbors
[12:32:06] WARNING: not removing hydrogen atom without neighbors
[12:32:06] WARNING: not removing hydrogen atom without neighbors
[12:32:06] WARNING: not removing hydrogen atom without neighbors
[12:32:06] WARNING: not removing hydrogen atom without neighbors
[12:32:06] WARNING: not removing hydrogen atom without neighbors
[12:32:07] WARNING: not removing hydrogen atom without neighbors
[12:32:07] WARNING: not removing hydrogen atom without neighbors
[12:32:07] WARNING: not removing hydrogen atom without neighbors
[12:32:07] WARNING: not removing hydrogen atom without neighbors
[12:32:07] WARNING: not removing hydrogen atom without neighbors
[12:32:07] WARNING: not removing hydrogen atom without neighbors
[12:32:07] WARNING: not r


Atom count ranges across 9117 molecules:
  C (Carbon):   min = 9, max = 34
  N (Nitrogen): min = 3, max = 11
  O (Oxygen):   min = 3, max = 33
  S (Sulfur):   min = 0, max = 5

Suggested max_atoms config (max values + 50% increase for expanded search space):
  C: 51
  N: 17
  O: 50
  S: 8


[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors
[12:32:08] WARNING: not removing hydrogen atom without neighbors


,recordIndex,PUBCHEM_COMPOUND_CID,PUBCHEM_COMPOUND_CANONICALIZED,PUBCHEM_CACTVS_COMPLEXITY,PUBCHEM_CACTVS_HBOND_ACCEPTOR,PUBCHEM_CACTVS_HBOND_DONOR,PUBCHEM_CACTVS_ROTATABLE_BOND,PUBCHEM_CACTVS_SUBSKEYS,PUBCHEM_IUPAC_OPENEYE_NAME,PUBCHEM_IUPAC_CAS_NAME,...,PUBCHEM_BONDANNOTATIONS,smiles,inchiKey,PUBCHEM_XLOGP3_AA,PUBCHEM_NONSTANDARDBOND,Canonical_SMILES,C,N,O,S
0,-1,None,None,None,None,None,None,None,None,None,...,None,C1=NC2=C(C(=O)N1)N=CN2[C@H]3[C@@H]([C@@H]([C@H...,UGQMRVRMYYASKQ-KQYNXXCUSA-N,None,None,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,10,4,5,0
1,0,135398640,1,555,10,5,4,AAADccBzvAIAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"[(2R,3S,4R,5R)-3,4-dihydroxy-5-(6-oxo-1H-purin...","[(2R,3S,4R,5R)-3,4-dihydroxy-5-(6-oxo-1H-purin...",...,16 10 6\n10 19 8\n10 20 8\n11 20 8\n11...,[H]O[C@@]1([H])[C@@]([H])(O[H])[C@]([H])(n2c([...,GRSZFWQUAKGDAV-KQYNXXCUSA-N,NaN,NaN,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)O)[C...,10,4,8,0
2,1,135398641,1,405,7,4,2,AAADccBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxymethy...","9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxymethy...",...,13 14 6\n15 17 8\n17 18 8\n10 2 5\n12 ...,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,UGQMRVRMYYASKQ-KQYNXXCUSA-N,NaN,NaN,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,10,4,5,0
3,2,135398739,1,348,5,2,2,AAADccBzsAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"9-[(2R,5S)-5-(hydroxymethyl)tetrahydrofuran-2-...","9-[(2R,5S)-5-(hydroxymethyl)-2-oxolanyl]-1H-pu...",...,11 12 6\n13 15 8\n15 16 8\n4 13 8\n4 ...,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,BXZVVICBKDXVGW-NKWVEPMBSA-N,NaN,NaN,O=c1[nH]cnc2c1ncn2[C@H]1CC[C@@H](CO)O1,10,4,3,0
4,3,135398635,1,446,7,5,2,AAADccBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"2-amino-9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydr...","2-amino-9-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydr...",...,14 15 5\n16 18 8\n18 19 8\n11 2 6\n13 ...,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,NYHBQMYGNKIUIF-UUOKFMHZSA-N,NaN,NaN,Nc1nc2c(ncn2[C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)...,10,5,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9112,9111,177857901,1,446,7,5,2,AAADccBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"2-amino-9-[(2S,3S,5S)-3,4-dihydroxy-5-(hydroxy...","2-amino-9-[(2S,3S,5S)-3,4-dihydroxy-5-(hydroxy...",...,14 15 6\n16 18 8\n18 19 8\n11 2 5\n13 ...,[H]OC([H])([H])[C@]1([H])O[C@]([H])(n2c([H])nc...,NYHBQMYGNKIUIF-IASZTVIBSA-N,NaN,NaN,Nc1nc2c(ncn2[C@H]2O[C@@H](CO)C(O)[C@@H]2O)c(=O...,10,5,5,0
9113,9112,177857919,1,587,9,6,5,AAADceBzuAAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"2-[[9-[(3R,4S,5R)-3,4-dihydroxy-5-(hydroxymeth...","2-[[9-[(3R,4S,5R)-3,4-dihydroxy-5-(hydroxymeth...",...,10 18 8\n10 22 8\n11 21 8\n11 22 8\n16...,[H]OC(=O)C([H])(N([H])c1nc2c(nc([H])n2C2([H])O...,JYXKXKDWVWQIOH-GZSQIZNNSA-N,NaN,NaN,CC(Nc1nc2c(ncn2C2O[C@H](CO)[C@@H](O)[C@H]2O)c(...,13,5,7,0
9114,9113,177857943,1,491,7,4,2,AAADccBzuQAAAAAAAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"2-amino-9-[(2R,3R,5R)-4-fluoro-3-hydroxy-5-(hy...","2-amino-9-[(2R,3R,5R)-4-fluoro-3-hydroxy-5-(hy...",...,13 1 3\n14 16 6\n17 19 8\n19 20 8\n11 ...,[H]OC([H])([H])[C@@]1([H])O[C@@]([H])(n2c([H])...,FWLVLYHLIABSLF-RBXWHLPPSA-N,NaN,NaN,C[C@]1(O)C(F)[C@@H](CO)O[C@H]1n1cnc2c(=O)[nH]c...,11,5,4,0
9115,9114,177860646,1,899,16,4,7,AAADccBzvAMAAAAEAAAAAAAAAAAAAWJAAAAgAAAAAAAAAE...,"[[[(2R,3S,4R,5R)-5-(2-amino-6-oxo-1H-purin-9-y...","[[[(2R,3S,4R,5R)-5-(2-amino-6-oxo-1H-purin-9-y...",...,26 19 5\n19 29 8\n19 30 8\n20 30 8\n20...,[Cu].[H]O[C@@]1([H])[C@@]([H])(O[H])[C@]([H])(...,JICNCLFSGJYKMS-GWTDSMLYSA-J,NaN,NaN,Nc1nc2c(ncn2[C@@H]2O[C@H](COP(=O)([O-])OP(=O)(...,10,5,14,0
